### Making A New GPKG for Community Lifelines
##### Use a Dictionary to take Sub-Sectors from CISA to FEMA categories
#### and Testing some Maps !

In [17]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import mapclassify
import contextily as ctx
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl
import os

In [ ]:
#read hex bin layer with all infrastrcuture merged together
gdf_merged = gpd.read_file(r'hex_infrastructure_merged.shp')


#make map of values by sub sector
#Community Lifelines listed second and Sub-Sector listed first
mapping_dict = {
            "Cellular Towers": "Communications",
            "Microwave Towers": "Communications",
            "Paging Transmission Towers": "Communications",
            "Land Mobile Boradcast Towers": "Communications",
            "Antenna Structures": "Communications",
            "Broadband": "Communications",

            "SNAP Retailers": "Food, Hydration, Shelter",
            "Nursing Care Facilities": "Health and Medical",
            "Food Banks": "Food, Hydration, Shelter",
            "Refrigerated Warehouse": "Food, Hydration, Shelter",
            "Retail Grocer": "Food, Hydration, Shelter",
            "Food Services": "Food, Hydration, Shelter",
            "Homeless Shelters": "Food, Hydration, Shelter",
            "Retirement Community": "Food, Hydration, Shelter",
            "Assisted Living": "Food, Hydration, Shelter",
            "Intermodal Locations": "Food, Hydration, Shelter",
            "Religious Organizations": "Food, Hydration, Shelter",

            "State Agency Buildings": "Safety and Security",
            "Public Schools": "Safety and Security",
            "Higher Education": "Safety and Security",
            "Public Libraries": "Safety and Security",
            "Private Schools": "Safety and Security",
            "USPS Plants": "Safety and Security",
            "Regional FBI Building": "Safety and Security",
            "Correctional Facilities": "Safety and Security",
            "Local Law Enforcement": "Safety and Security",
            "Fire Stations": "Safety and Security",
            "VDEM Regional Office": "Safety and Security",
            "VDEM State Office": "Safety and Security",
            "State EOC": "Safety and Security",
            "Local Emergency Shelters": "Safety and Security",
            "Public Works Departments": "Safety and Security",
            "Dry Hydrants": "Safety and Security",
            "Local EOC": "Safety and Security",
            "Courthouses": "Safety and Security",
            "Banks": "Safety and Security",

            "Amtrak Stations": "Transportation",   
            "Aviation Locations": "Transportation",
            "Rail Bridges": "Transportation",
            "Rail Crossings": "Transportation",
            "Bus Stops": "Transportation",
            "T_Mi_Index": "Transportation",
            "RO/RO Freight": "Transportation",

            "Alternative Fueling Stations": "Energy (Power & Fuel)",
            "Biodiesel Plant": "Energy (Power & Fuel)",
            "Electric Substations or Switching Stations": "Energy (Power & Fuel)",
            "Ethanol Plant": "Energy (Power & Fuel)",
            "Ethanol Transloading": "Energy (Power & Fuel)",
            "LNG Storage": "Energy (Power & Fuel)",
            "Natural Gas Compressor Stations": "Energy (Power & Fuel)",
            "Petroleum Port": "Energy (Power & Fuel)",
            "Petroleum Terminals": "Energy (Power & Fuel)",
            "Petroleum Pumping Stations": "Energy (Power & Fuel)",
            "Battery-Energy Storage Stations": "Energy (Power & Fuel)",
            "Gas and Oil Plugged Wells": "Energy (Power & Fuel)",
            "Power Plants": "Energy (Power & Fuel)",
            "E_Mi_Index": "Energy (Power & Fuel)",

            "School Bus Storage": "Hazardous Materials",
            "Waste and Recycling Facilities": "Hazardous Materials",
            "Debris Storage": "Hazardous Materials",

            "Water Point Data": "Water Systems",
            "Sewer Point Data": "Water Systems",
            "Wastewater Treatment": "Water Systems",
            "Water Storage Tanks": "Water Systems",
            "Public Drinking Water": "Water Systems",

            "Ambulance Services": "Health and Medical",
            "Diagnostic Imaging": "Health and Medical",
            "Dialysis Center": "Health and Medical",
            "Disability Facilities": "Health and Medical",
            "Family Reunification Center": "Health and Medical",
            "Family Services": "Health and Medical",
            "Health Education": "Health and Medical",
            "Health Pactitioners": "Health and Medical",
            "Home Health Care": "Health and Medical",
            "Hospitals": "Health and Medical",
            "Medical Laboratories": "Health and Medical",
            "Mental Health": "Health and Medical",
            "Outpatient Care Centers": "Health and Medical",
            "Outpatient Mental Health": "Health and Medical",
            "Pharmacies": "Health and Medical",
            "Psychiatric Hospitals": "Health and Medical",
            "Public Health Departments": "Health and Medical",
            "Red Cross": "Health and Medical",
            "Residential Care Facilities": "Health and Medical",
            "Specialty Hospitals": "Health and Medical",
            "Evacuation Assembly Center": "Health and Medical"
        }

# Group columns by lifeline
lifeline_groups = defaultdict(list)
for sub_sector, lifeline in mapping_dict.items():
    if sub_sector in gdf_merged.columns:
        lifeline_groups[lifeline].append(sub_sector)

# Create new columns with summed values
for lifeline, cols in lifeline_groups.items():
    gdf_merged[lifeline] = gdf_merged[cols].sum(axis=1)

# fill NaNs with 0 if needed
created_cols = list(lifeline_groups.keys())

if created_cols:
    gdf_merged[created_cols] = gdf_merged[created_cols].fillna(0)


# Desired output columns
desired_columns = [
    "Communications",
    "Food, Hydration, Shelter",
    "Safety and Security",
    "Water Systems",
    "Health and Medical",
    "Transportation",
    "Energy (Power & Fuel)",
    "Hazardous Materials",
    "geometry",
    "h3_ID"
]

# Keep only columns that actually exist
existing_columns = [col for col in desired_columns if col in gdf_merged.columns]

gdf_merged = gdf_merged[existing_columns]

# Check column names
column_names_list = gdf_merged.columns.tolist()
print(column_names_list)

# Save output to H3 Edits folder
output_path = r"\hex_community_lifelines.gpkg"

gdf_merged.to_file(output_path, driver="GPKG")


#### Trying to fix the integer problem

In [ ]:
#read  layer
hex_path = "\hex_community_lifelines.gpkg"
#Define GPD:
gdf_merged = gpd.read_file(hex_path)
cols_to_int = ["Communications", "Food, Hydration, Shelter", "Safety and Security", "Water Systems",
    "Health and Medical", "Transportation", "Energy (Power & Fuel)", "Hazardous Materials"]  

# Convert and handles NaNs
for col in cols_to_int:
    # Convert everything to numeric (invalid → NaN)
    gdf_merged[col] = pd.to_numeric(gdf_merged[col], errors="coerce")
    
    # Optional: round if needed
    gdf_merged[col] = gdf_merged[col].round()
    
    # Convert to nullable integer
    gdf_merged[col] = gdf_merged[col].astype("Int64")

#save and overwrite with additions
gdf_merged.to_file("\hex_community_lifelines.gpkg", driver="GPKG")

#### Making Total Column, and Stretching

In [ ]:
#had to re-read here and fix the normlization column
hex_path = "\hex_community_lifelines.gpkg"
#Define GPD:
gdf_merged = gpd.read_file(hex_path)

cols = [
    "Communications",
    "Food, Hydration, Shelter",
    "Safety and Security",
    "Water Systems",
    "Health and Medical",
    "Transportation",
    "Energy (Power & Fuel)",
    "Hazardous Materials"
]

# Ensure numeric (invalid values → NaN)
gdf_merged[cols] = gdf_merged[cols].apply(pd.to_numeric, errors="coerce")

# Create total column (skip NaNs automatically)
gdf_merged["total"] = gdf_merged[cols].sum(axis=1)
gdf_merged["total"] = gdf_merged["total"].fillna(0)

gdf_merged["total"] = gdf_merged["total"].round().astype("Int64")

# Min-max normalization (0–1)
min_val = gdf_merged["total"].min()
max_val = gdf_merged["total"].max()

if max_val == min_val:
    gdf_merged["Total_Index"] = 100
else:
    gdf_merged["Total_Index"] = (
        (gdf_merged["total"] - min_val) / (max_val - min_val)
    ) * 99 + 1

print(gdf_merged["Total_Index"].describe())

#save and overwrite with additions
gdf_merged.to_file("\hex_community_lifelines.gpkg", driver="GPKG")

In [9]:
#checking column totals
totals = gdf_merged.select_dtypes(include="number").sum()

cols = ["Communications", "Food, Hydration, Shelter",
    "Safety and Security",
    "Water Systems",
    "Health and Medical",
    "Transportation",
    "Energy (Power & Fuel)",
    "Hazardous Materials"]
totals = gdf_merged[cols].sum()

totals_df = totals.reset_index()
totals_df.columns = ["Category", "Total"]
totals_df.to_csv("selected_totals2.csv", index=False)

In [ ]:
#Checking the information on a map! Here is an example of using .explore

#PlanRVA font and colors
mpl.rcParams["font.family"] = "Montserrat"

#the colormap for this project 
custom_cmap = LinearSegmentedColormap.from_list(
    "custom_blue",
    ["#e8e8e8", "#a092c1", "#7d68ae"]  # light → medium → dark
)

# get the right crs for OSM basemap
gdf_web = gdf_merged.to_crs(epsg=3857)


fig, ax = plt.subplots(1, 1, figsize=(10, 10))

gdf_web.plot(
    column="Total_Index",
    cmap=custom_cmap,
    linewidth=0,    
    edgecolor="none",
    legend=True,
    ax=ax,
    legend_kwds={
        "label": "Infrastructure Count",
        "orientation": "vertical",
        "shrink": 0.5 
    }
)

ctx.add_basemap(
    ax,
    source=ctx.providers.OpenStreetMap.Mapnik
)


ax.set_title("Infrastructure by Hex", fontsize=16)
ax.axis("off")
plt.tight_layout()
plt.show()